In [1]:
# ==============================================================================
# 1. CONFIGURAÇÃO DO AMBIENTE E IMPORTAÇÕES
# ==============================================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_samples, silhouette_score

# Montar Google Drive (descomente caso esteja no Google Colab)
# from google.colab import drive
# drive.mount('/content/drive')

# Configuração do diretório de trabalho
CAMINHO_PASTA = '/content/drive/MyDrive/Colab Notebooks'

if os.path.exists(CAMINHO_PASTA):
    os.chdir(CAMINHO_PASTA)
    print("Diretório atual:", os.getcwd())
else:
    print("Aviso: Caminho especificado não encontrado. Mantendo diretório atual:", os.getcwd())


# ==============================================================================
# 2. CARREGAMENTO DOS DATASETS
# ==============================================================================
def carregar_dados(diretorio="."):
    """Carrega os arquivos CSV do projeto com os parâmetros pré-definidos."""
    caminhos = {
        "Data_set": os.path.join(diretorio, "Data_set.CSV"),
        "DF": os.path.join(diretorio, "Dataset_Dimensional_Final.csv"),
        "TP": os.path.join(diretorio, "Dataset_Transformacoes_Pi.csv"),
        "MP": os.path.join(diretorio, "Dataset_Modelagem_Pi.csv")
    }

    dataframes = {}
    for nome, caminho in caminhos.items():
        if os.path.exists(caminho):
            dataframes[nome] = pd.read_csv(caminho, sep=";", decimal=",", encoding="latin-1")
            print(f"[✓] {nome} carregado com sucesso.")
        else:
            print(f"[!] Arquivo não encontrado: {caminho}")

    return dataframes

# Executar carregamento
dados = carregar_dados()
Data_set = dados.get("Data_set")
DF = dados.get("DF")


# ==============================================================================
# 3. FUNÇÕES AUXILIARES DE PRÉ-PROCESSAMENTO E VISUALIZAÇÃO (SEM SEABORN)
# ==============================================================================
def preparar_dados_clustering(df, colunas_features):
    """
    Filtra as colunas solicitadas, trata valores nulos e aplica padronização (StandardScaler).
    Retorna o dataframe tratado e a matriz/DataFrame escalados.
    """
    df_sub = df[colunas_features].copy()

    # Imputação de nulos pela média
    if df_sub.isnull().sum().sum() > 0:
        df_sub.fillna(df_sub.mean(), inplace=True)

    # Removendo colunas que permaneceram totalmente nulas (se houver)
    cols_validas = df_sub.columns[~df_sub.isnull().all()]
    df_sub = df_sub[cols_validas]

    scaler = StandardScaler()
    dados_escalados = scaler.fit_transform(df_sub)
    df_escalado = pd.DataFrame(dados_escalados, columns=df_sub.columns)

    return df_sub, df_escalado, scaler


def rodar_elbow_kmeans(df_scaled, max_k=10, titulo="Método do Cotovelo"):
    """Gera o gráfico do cotovelo para avaliação do K ideal."""
    wcss = []
    for k in range(1, max_k + 1):
        kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=max_k, random_state=42)
        kmeans.fit(df_scaled)
        wcss.append(kmeans.inertia_)

    plt.figure(figsize=(9, 5))
    plt.plot(range(1, max_k + 1), wcss, marker='o', linestyle='--', color='tab:blue', alpha=0.9)
    plt.title(titulo)
    plt.xlabel('Número de Clusters (K)')
    plt.ylabel('WCSS')
    plt.xticks(range(1, max_k + 1))
    plt.grid(True)
    plt.show()


def rodar_k_distance_dbscan_log(df_scaled, min_samples, max_y=None, titulo="K-Distance Graph (Escala Log)"):
    """Gera o gráfico de K-Distância em escala logarítmica para facilitar a visualização do joelho."""
    neighbors = NearestNeighbors(n_neighbors=min_samples)
    neighbors_fit = neighbors.fit(df_scaled)
    distances, _ = neighbors_fit.kneighbors(df_scaled)
    distances = np.sort(distances[:, min_samples - 1], axis=0)

    plt.figure(figsize=(10, 5))
    plt.plot(distances, color='tab:blue', alpha=0.9)
    plt.yscale('log') # Transforma o eixo Y em escala logarítmica
    plt.title(f"{titulo} (min_samples = {min_samples})")
    plt.xlabel('Pontos Ordenados')
    plt.ylabel(f'Distância para o {min_samples}º Vizinho (Escala Log)')
    plt.grid(True, which="both", linestyle='--', alpha=0.5)
    if max_y:
        plt.ylim(top=max_y)
    plt.show()

def avaliar_grid_dbscan(df_scaled, min_samples, eps_range=np.arange(0.1, 3.0, 0.1)):
    """
    Plota um diagnóstico com a taxa de ruído e o número de clusters gerados
    para diferentes valores de eps.
    """
    resultados = []

    for eps in eps_range:
        db = DBSCAN(eps=eps, min_samples=min_samples).fit(df_scaled)
        labels = db.labels_

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        pct_ruido = (labels == -1).sum() / len(labels) * 100

        resultados.append({
            'eps': eps,
            'n_clusters': n_clusters,
            'pct_ruido': pct_ruido
        })

    res_df = pd.DataFrame(resultados)

    fig, ax1 = plt.subplots(figsize=(10, 5))

    # Curva 1: Percentual de Ruído
    color = 'tab:red'
    ax1.set_xlabel('eps (Raio de Vizinhança)')
    ax1.set_ylabel('% de Ruído (Outliers)', color=color)
    ax1.plot(res_df['eps'], res_df['pct_ruido'], color=color, marker='o', linewidth=2)
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, linestyle=':', alpha=0.6)

    # Curva 2: Número de Clusters
    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Nº de Clusters Encontrados', color=color)
    ax2.plot(res_df['eps'], res_df['n_clusters'], color=color, marker='s', linestyle='--', linewidth=2)
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title(f'Diagnóstico de DBSCAN (min_samples = {min_samples})', pad=15)
    fig.tight_layout()
    plt.show()


def visualizar_clusters_pca(df_scaled, clusters, titulo="Visualização PCA", xlim=None, ylim=None):
    """
    Reduz os dados escalados para 2D via PCA e plota os clusters usando apenas Matplotlib.
    """
    pca = PCA(n_components=2)
    componentes = pca.fit_transform(df_scaled)

    plt.figure(figsize=(10, 6))

    scatter = plt.scatter(
        componentes[:, 0],
        componentes[:, 1],
        c=clusters,
        cmap='viridis',
        s=80,
        alpha=0.9
    )

    legend = plt.legend(*scatter.legend_elements(), title="Cluster", loc="upper right")
    plt.gca().add_artist(legend)

    plt.title(titulo)
    plt.xlabel('Componente Principal 1')
    plt.ylabel('Componente Principal 2')
    if xlim: plt.xlim(xlim)
    if ylim: plt.ylim(ylim)
    plt.grid(True)
    plt.show()


def plotar_silhouette_analysis(df_scaled, cluster_labels, titulo="Análise de Silhueta"):
    """
    Gera o gráfico de Silhueta individual para cada cluster, juntamente com
    a linha tracejada correspondente ao Silhouette Score médio global.
    """
    clusters_validos = np.unique(cluster_labels[cluster_labels != -1])
    n_clusters = len(clusters_validos)

    if n_clusters < 2:
        print(f"[{titulo}] Não é possível gerar a Análise de Silhueta com menos de 2 clusters válidos.")
        return

    silhouette_avg = silhouette_score(df_scaled, cluster_labels)
    sample_silhouette_values = silhouette_samples(df_scaled, cluster_labels)

    fig, ax = plt.subplots()#figsize=(10, 6))

    y_lower = 10
    colors = plt.cm.viridis(np.linspace(0, 1, n_clusters))

    for i, cluster_id in enumerate(clusters_validos):
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == cluster_id]
        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = colors[i % len(colors)]
        ax.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
            label=f'Cluster {cluster_id}'
        )

        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(cluster_id), va='center', ha='right', fontsize=10, fontweight='bold')
        y_lower = y_upper + 10

    ax.axvline(x=silhouette_avg, color="red", linestyle="--", linewidth=2, label=f'Média Global ({silhouette_avg:.3f})')

    ax.set_title(f"{titulo}\n(Silhouette Score Médio: {silhouette_avg:.3f})", fontsize=14, pad=15)
    ax.set_xlabel("Coeficiente de Silhueta", fontsize=12)
    ax.set_ylabel("Clusters (amostras ordenadas)", fontsize=12)

    ax.set_xlim([-0.1, 1.0])
    ax.set_yticks([])
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper right')

    plt.tight_layout()
    plt.show()


# ==============================================================================
# 4. EXECUÇÃO NO DATAFRAME `DF`
# ==============================================================================
if DF is not None:
    print("\n" + "="*50)
    print(" INICIANDO ANÁLISE NO DATAFRAME 'DF'")
    print("="*50)

    # 4.1 Seleção de Features
    pi_cols = [c for c in DF.columns if 'Pi_' in c and DF[c].dtype != 'object']
    features_df = ['Temp'] + pi_cols

    df_clustering_df, df_scaled_df, _ = preparar_dados_clustering(DF, features_df)

    # 4.2 Avaliação do K (Elbow Method)
    rodar_elbow_kmeans(df_scaled_df,max_k=15, titulo="Cotovelo - DataFrame DF")

    # 4.3 Treinamento do K-Means (Ajustado K de exemplo para um valor razoável para visualização)
    k_escolhido = 4 # cotovelo fica entorno de 4 (verificar pela tangente ou regua)
    kmeans_df = KMeans(n_clusters=k_escolhido, init='k-means++', max_iter=300, n_init=15, random_state=42)
    DF['Cluster_KMeans'] = kmeans_df.fit_predict(df_scaled_df)

    # Visualizar PCA K-Means
    visualizar_clusters_pca(df_scaled_df, DF['Cluster_KMeans'], f'DF - K-Means (K={k_escolhido})', xlim=(-0.5, 0.5), ylim=(-3.5, 5))

    # Análise de Silhueta ao invés do Radar Chart
    plotar_silhouette_analysis(df_scaled_df, DF['Cluster_KMeans'].values, f'Silhueta K-Means (K={k_escolhido}) - DF')

    # 4.4 Avaliação e Execução do DBSCAN
    min_samples_df = 122 # Encontra o cotovelo


    rodar_k_distance_dbscan_log(df_scaled_df, min_samples=min_samples_df, max_y=25, titulo="DBSCAN K-Distance - DF")

    # Testando com um valor de eps
    #avaliar_grid_dbscan(df_scaled_df, min_samples_df, eps_range=np.arange(1.55, 1.57, 0.001))
    eps_escolhido = 1.56 # Encontra a estabilidade (erro <= 10%)

    dbscan_df = DBSCAN(eps=eps_escolhido, min_samples=min_samples_df)
    DF[f'Cluster_DBSCAN_{eps_escolhido}'] = dbscan_df.fit_predict(df_scaled_df)



    # Visualizar PCA DBSCAN
    visualizar_clusters_pca(df_scaled_df, DF[f'Cluster_DBSCAN_{eps_escolhido}'], f'DF - DBSCAN (eps={eps_escolhido})', xlim=(-2, 5), ylim=(-3.5, 5))

    # Silhueta no DBSCAN (Opcional, ignora ruídos -1)
    plotar_silhouette_analysis(df_scaled_df, DF[f'Cluster_DBSCAN_{eps_escolhido}'].values, f'Silhueta DBSCAN (eps={eps_escolhido}) - DF')

    print(f"Contagem de pontos por cluster (eps={eps_escolhido}):")

    contagem = DF[f'Cluster_DBSCAN_{eps_escolhido}'].value_counts()
    proporcao = DF[f'Cluster_DBSCAN_{eps_escolhido}'].value_counts(normalize=True)
    tabela_resumo1 = pd.DataFrame({'Quantidade': contagem, 'Porcentagem': proporcao})
    display(tabela_resumo1.style.format({'Porcentagem': '{:.2%}'}))


# ==============================================================================
# 5. EXECUÇÃO NO DATAFRAME `Data_set`
# ==============================================================================
if Data_set is not None:
    print("\n" + "="*50)
    print(" INICIANDO ANÁLISE NO DATAFRAME 'Data_set'")
    print("="*50)

    # 5.1 Seleção de Features Numéricas
    df_num = Data_set.select_dtypes(include=np.number).copy()

    # Remover coluna temporal/índice se presente
    col_tempo = 'D1T1A - Tempo (amostras por segundo) [s]'
    if col_tempo in df_num.columns:
        df_num.drop(columns=[col_tempo], inplace=True)

    df_clustering_dataset, df_scaled_dataset, _ = preparar_dados_clustering(Data_set, df_num.columns)

    # 5.2 Avaliação do K (Elbow Method)
    rodar_elbow_kmeans(df_scaled_dataset, titulo="Cotovelo - Data_set")

    # 5.3 K-Means no Data_set
    k_ds = 4
    kmeans_ds = KMeans(n_clusters=k_ds, init='k-means++', max_iter=300, n_init=10, random_state=42)
    clusters_ds = kmeans_ds.fit_predict(df_scaled_dataset)
    Data_set['Cluster_KMeans'] = clusters_ds

    visualizar_clusters_pca(df_scaled_dataset, clusters_ds, f'Data_set - K-Means (K={k_ds})', xlim=(-10, 10), ylim=(-3, 3))
    plotar_silhouette_analysis(df_scaled_dataset, clusters_ds, f'Silhueta K-Means (K={k_ds}) - Data_set')

    # 5.4 DBSCAN no Data_set
    min_samples_ds = 7 #df_scaled_dataset.shape[1]
    rodar_k_distance_dbscan_log(df_scaled_dataset, min_samples=min_samples_ds, max_y=10, titulo="DBSCAN K-Distance - Data_set")

    #avaliar_grid_dbscan(df_scaled_dataset, min_samples_ds, eps_range=np.arange(6, 8, 0.1))
    eps_ds = 6.65
    dbscan_ds = DBSCAN(eps=eps_ds, min_samples=min_samples_ds)
    Data_set['Cluster_DBSCAN'] = dbscan_ds.fit_predict(df_scaled_dataset)

    visualizar_clusters_pca(df_scaled_dataset, Data_set['Cluster_DBSCAN'], f'Data_set - DBSCAN (eps={eps_ds})', xlim=(-10, 10), ylim=(-3, 3))
    plotar_silhouette_analysis(df_scaled_dataset, Data_set[f'Cluster_DBSCAN'].values, f'Silhueta DBSCAN (eps={eps_ds}) - DF')

    contagem = Data_set['Cluster_DBSCAN'].value_counts()
    proporcao = Data_set['Cluster_DBSCAN'].value_counts(normalize=True)
    tabela_resumo2 = pd.DataFrame({'Quantidade': contagem, 'Porcentagem': proporcao})
    display(tabela_resumo2.style.format({'Porcentagem': '{:.2%}'}))



Aviso: Caminho especificado não encontrado. Mantendo diretório atual: /home/henriquesilva/phd/machine-learning-time-series/notebooks
[!] Arquivo não encontrado: ./Data_set.CSV
[!] Arquivo não encontrado: ./Dataset_Dimensional_Final.csv
[!] Arquivo não encontrado: ./Dataset_Transformacoes_Pi.csv
[!] Arquivo não encontrado: ./Dataset_Modelagem_Pi.csv
